In [3]:
import sys
import os
import torch
import pandas as pd
from pathlib import Path

# Adiciona a pasta 'Codigo' ao sys.path seguindo o padrão que já funcionou nos seus outros testes
sys.path.append(str(Path.cwd().parent / "Codigo"))

from model import GPTModel


# ============================================================
# Teste de Escalabilidade e Custo Computacional do GPT
# ============================================================

def rodar_experimentos():
    VOCAB_SIZE = 50257
    CONTEXT_LENGTH = 1024
    DROP_RATE = 0.1
    QKV_BIAS = False

    gpt_configs = {
        "GPT-2 Small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
        "GPT-2 Medium (345M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
        "GPT-2 Large (762M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
        "GPT-2 XL (1.5B)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
    }

    resultados = []
    print("Iniciando a simulação de carga...\n")

    for model_name, cfg in gpt_configs.items():
        current_cfg = {
            "vocab_size": VOCAB_SIZE,
            "context_length": CONTEXT_LENGTH,
            "drop_rate": DROP_RATE,
            "qkv_bias": QKV_BIAS,
            **cfg
        }
        
        # Instancia o modelo
        model = GPTModel(current_cfg)
        
        # Cálculo do número total de parâmetros
        total_params = sum(p.numel() for p in model.parameters())
        params_out_head = sum(p.numel() for p in model.out_head.parameters())
        total_params_ajustado = total_params - params_out_head
        
        # Cálculo de memória estimada em MB (float32 -> 4 bytes)
        total_size_bytes = total_params_ajustado * 4
        total_size_mb = total_size_bytes / (1024 * 1024)
        
        resultados.append({
            "Versão": model_name,
            "Dimensão (emb_dim)": cfg["emb_dim"],
            "Camadas (n_layers)": cfg["n_layers"],
            "Cabeças (n_heads)": cfg["n_heads"],
            "Parâmetros Treináveis": f"{total_params_ajustado:,}",
            "Tamanho Memória (MB)": round(total_size_mb, 2)
        })
        
        del model # Limpa a memória RAM entre as iterações

    # Renderiza a tabela no Jupyter Notebook
    df_resultados = pd.DataFrame(resultados)
    print("=== RESULTADO DA ANÁLISE DE ESCALABILIDADE ===")
    display(df_resultados)

# Executa o teste
rodar_experimentos()

Iniciando a simulação de carga...

=== RESULTADO DA ANÁLISE DE ESCALABILIDADE ===


,Versão,Dimensão (emb_dim),Camadas (n_layers),Cabeças (n_heads),Parâmetros Treináveis,Tamanho Memória (MB)
0,GPT-2 Small (124M),768,12,12,"124,412,160",474.59
1,GPT-2 Medium (345M),1024,24,16,"354,749,440",1353.26
2,GPT-2 Large (762M),1280,36,20,"773,891,840",2952.16
3,GPT-2 XL (1.5B),1600,48,25,"1,557,380,800",5940.94


In [4]:
import torch
# Adiciona a pasta 'Codigo' ao sys.path seguindo o padrão que já funcionou nos seus outros testes
sys.path.append(str(Path.cwd().parent / "Codigo"))
from model import TransformerBlock # Ajuste a importação se necessário

# 1. Configuração encapsulada (cfg) correspondente ao seu __init__
cfg = {
    "emb_dim": 64,
    "context_length": 128,
    "n_heads": 4,
    "drop_rate": 0.0,
    "qkv_bias": False
}
batch_size = 2
seq_len = 8

print("=== Análise Trilha de Tensor: Transformer Block ===")

# 2. Instanciar o bloco passando o dicionário exigido
block = TransformerBlock(cfg)
block.eval() 

# 3. Gerar tensor de entrada fictício 
x = torch.randn(batch_size, seq_len, cfg["emb_dim"])
print(f"1. Entrada (x)                    : {list(x.shape)} -> [batch, seq_len, emb_dim]")

with torch.no_grad():
    # Etapa 1: Pre-Norm + Attention + Add (com drop_shortcut)
    norm1_out = block.norm1(x) 
    print(f"2. Saída LayerNorm 1              : {list(norm1_out.shape)}")
    
    attn_out = block.att(norm1_out) 
    print(f"3. Saída Multi-Head Attention     : {list(attn_out.shape)}")
    
    x_res_1 = x + block.drop_shortcut(attn_out)
    print(f"4. 1ª Conexão Residual            : {list(x_res_1.shape)}")
    
    # Etapa 2: Pre-Norm + FFN + Add
    norm2_out = block.norm2(x_res_1) 
    print(f"5. Saída LayerNorm 2              : {list(norm2_out.shape)}")
    
    ffn_out = block.ff(norm2_out)
    print(f"6. Saída FeedForward Network      : {list(ffn_out.shape)}")
    
    out_final = x_res_1 + block.drop_shortcut(ffn_out)
    print(f"7. 2ª Conexão Residual (Final)    : {list(out_final.shape)}")

# Asserção rigorosa
assert out_final.shape == x.shape, "Erro na arquitetura: Truncamento ou perda dimensional detectada."
print("\n[OK] Validação estrutural concluída: Integridade do bloco garantida.")

=== Análise Trilha de Tensor: Transformer Block ===
1. Entrada (x)                    : [2, 8, 64] -> [batch, seq_len, emb_dim]
2. Saída LayerNorm 1              : [2, 8, 64]
3. Saída Multi-Head Attention     : [2, 8, 64]
4. 1ª Conexão Residual            : [2, 8, 64]
5. Saída LayerNorm 2              : [2, 8, 64]
6. Saída FeedForward Network      : [2, 8, 64]
7. 2ª Conexão Residual (Final)    : [2, 8, 64]

[OK] Validação estrutural concluída: Integridade do bloco garantida.


In [5]:
import torch
# Adiciona a pasta 'Codigo' ao sys.path seguindo o padrão que já funcionou nos seus outros testes
sys.path.append(str(Path.cwd().parent / "Codigo"))
from model import GPTModel # Certifique-se que o GPTModel está salvo em model.py

# 1. Dicionário de configuração da arquitetura GPT
cfg = {
    "vocab_size": 50257,     # Tamanho do vocabulário (ex: BPE GPT-2)
    "context_length": 256,   # Comprimento máximo de contexto suportado
    "emb_dim": 128,          # Dimensão do embedding
    "n_heads": 4,            # Número de cabeças de atenção
    "n_layers": 2,           # Número de blocos Transformer empilhados
    "drop_rate": 0.0,        # Desativado para auditoria determinística
    "qkv_bias": False
}

batch_size = 2
seq_len = 10

print("=== Análise Trilha de Tensor: Arquitetura GPT Completa ===")

# 2. Instanciar o modelo
model = GPTModel(cfg)
model.eval()

# 3. Gerar lote de IDs de tokens fictícios (valores entre 0 e vocab_size-1)
in_idx = torch.randint(0, cfg["vocab_size"], (batch_size, seq_len))
print(f"1. Entrada (Token IDs)           : {list(in_idx.shape)} -> [batch_size, seq_len]")

with torch.no_grad():
    # Passo a passo interno para auditoria
    tok_emb = model.tok_emb(in_idx)
    pos_emb = model.pos_emb(torch.arange(seq_len))
    x_emb = tok_emb + pos_emb
    print(f"2. Embeddings (Token + Posição)  : {list(x_emb.shape)} -> [batch_size, seq_len, emb_dim]")
    
    x_trf = model.trf_blocks(x_emb)
    print(f"3. Saída Empilhada (Trf Blocks)  : {list(x_trf.shape)} -> [batch_size, seq_len, emb_dim]")
    
    x_norm = model.final_norm(x_trf)
    print(f"4. Saída LayerNorm Final        : {list(x_norm.shape)} -> [batch_size, seq_len, emb_dim]")
    
    logits = model(in_idx)
    print(f"5. Saída Terminal (Logits)      : {list(logits.shape)} -> [batch_size, seq_len, vocab_size]")

# Asserção de integridade da saída terminal
expected_shape = (batch_size, seq_len, cfg["vocab_size"])
assert logits.shape == torch.Size(expected_shape), f"Erro! Esperado {expected_shape}, recebido {logits.shape}"

print("\n[OK] Validação da Arquitetura GPT concluída: Mapeamento de Token IDs para Logits ativo.")

=== Análise Trilha de Tensor: Arquitetura GPT Completa ===
1. Entrada (Token IDs)           : [2, 10] -> [batch_size, seq_len]
2. Embeddings (Token + Posição)  : [2, 10, 128] -> [batch_size, seq_len, emb_dim]
3. Saída Empilhada (Trf Blocks)  : [2, 10, 128] -> [batch_size, seq_len, emb_dim]
4. Saída LayerNorm Final        : [2, 10, 128] -> [batch_size, seq_len, emb_dim]
5. Saída Terminal (Logits)      : [2, 10, 50257] -> [batch_size, seq_len, vocab_size]

[OK] Validação da Arquitetura GPT concluída: Mapeamento de Token IDs para Logits ativo.


In [6]:
import torch
# Adiciona a pasta 'Codigo' ao sys.path seguindo o padrão que já funcionou nos seus outros testes
sys.path.append(str(Path.cwd().parent / "Codigo"))
from model import GPTModel
# Caso as funções estejam em generation.py: from generation import generate_text_sampling

# 1. Configuração da Arquitetura
cfg = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.0,
    "qkv_bias": False
}

model = GPTModel(cfg)
model.eval()

# 2. Definir sequência inicial (Prompt fictício de 4 tokens)
prompt_tokens = torch.tensor([[101, 2054, 2003, 1037]]) # [batch_size=1, seq_len=4]
max_new_tokens = 5

print("=== Validação do Ciclo Autorregressivo ===")
print(f"Contexto Inicial  : {prompt_tokens.tolist()[0]} | Tamanho: {prompt_tokens.size(1)}")

contexto = prompt_tokens.clone()

for passo in range(1, max_new_tokens + 1):
    with torch.no_grad():
        # Passagem restrita pela janela de contexto
        idx_cond = contexto[:, -cfg["context_length"]:]
        logits = model(idx_cond)
        
        # Isolar os logits do último token
        logits_ultimo = logits[:, -1, :] # Dimensão: [1, 50257]
        
        # Amostragem com temperatura e top-k
        temperature = 0.8
        top_k = 10
        
        top_logits, _ = torch.topk(logits_ultimo, top_k)
        logits_filtrados = torch.where(
            logits_ultimo < top_logits[:, -1],
            torch.tensor(float('-inf')),
            logits_ultimo
        )
        
        probs = torch.softmax(logits_filtrados / temperature, dim=-1)
        prox_token = torch.multinomial(probs, num_samples=1)
        
        # Concatenação autorregressiva
        contexto = torch.cat((contexto, prox_token), dim=1)
        
        print(f"Passo {passo} | Token Gerado: {prox_token.item():<6} | Novo Contexto: {contexto.tolist()[0]}")

print("\n[OK] Inferência Autorregressiva concluída: Concatenação iterativa validada.")

=== Validação do Ciclo Autorregressivo ===
Contexto Inicial  : [101, 2054, 2003, 1037] | Tamanho: 4
Passo 1 | Token Gerado: 185    | Novo Contexto: [101, 2054, 2003, 1037, 185]
Passo 2 | Token Gerado: 18782  | Novo Contexto: [101, 2054, 2003, 1037, 185, 18782]
Passo 3 | Token Gerado: 10114  | Novo Contexto: [101, 2054, 2003, 1037, 185, 18782, 10114]
Passo 4 | Token Gerado: 43148  | Novo Contexto: [101, 2054, 2003, 1037, 185, 18782, 10114, 43148]
Passo 5 | Token Gerado: 28009  | Novo Contexto: [101, 2054, 2003, 1037, 185, 18782, 10114, 43148, 28009]

[OK] Inferência Autorregressiva concluída: Concatenação iterativa validada.


In [7]:
import time
import torch
# Adiciona a pasta 'Codigo' ao sys.path seguindo o padrão que já funcionou nos seus outros testes
sys.path.append(str(Path.cwd().parent / "Codigo"))
from model import GPTModel
from generation import generate_text_advanced

def rodar_experimento_bench(nome_exp, cfg_exp, prompt_tensor):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. Instanciação e medição de parâmetros
    model = GPTModel(cfg_exp).to(device)
    model.eval()
    
    params_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
    memoria_mb = (params_treinaveis * 4) / (1024 * 1024) # Estimativa de memória base (float32)
    
    # 2. Medição de tempo de geração autorregressiva (10 tokens)
    input_idx = prompt_tensor.to(device)
    
    inicio = time.time()
    with torch.no_grad():
        out = generate_text_advanced(
            model=model,
            idx=input_idx,
            max_new_tokens=10,
            context_size=cfg_exp["context_length"],
            temperature=cfg_exp.get("temp", 0.8),
            top_k=cfg_exp.get("top_k", 10)
        )
    fim = time.time()
    tempo_exec = fim - inicio
    
    print(f"=== {nome_exp} ===")
    print(f"Parâmetros Treináveis : {params_treinaveis:,}")
    print(f"Memória Estimada Modelo: {memoria_mb:.2f} MB")
    print(f"Tempo de Inferência   : {tempo_exec:.4f} s\n")

# Configurações de teste
prompt = torch.tensor([[101, 2054, 2003, 1037]])

cfg_base = {"vocab_size": 50257, "context_length": 256, "emb_dim": 128, "n_heads": 4, "n_layers": 2, "drop_rate": 0.0, "qkv_bias": False}
cfg_deep = {"vocab_size": 50257, "context_length": 256, "emb_dim": 128, "n_heads": 4, "n_layers": 6, "drop_rate": 0.0, "qkv_bias": False}
cfg_wide = {"vocab_size": 50257, "context_length": 256, "emb_dim": 256, "n_heads": 8, "n_layers": 2, "drop_rate": 0.0, "qkv_bias": False}

rodar_experimento_bench("Exp 1: Base (Small)", cfg_base, prompt)
rodar_experimento_bench("Exp 2: Aumento de Layers (6 blocos)", cfg_deep, prompt)
rodar_experimento_bench("Exp 3: Aumento de Dimensão (256 emb / 8 heads)", cfg_wide, prompt)

=== Exp 1: Base (Small) ===
Parâmetros Treináveis : 13,294,592
Memória Estimada Modelo: 50.71 MB
Tempo de Inferência   : 0.0728 s

=== Exp 2: Aumento de Layers (6 blocos) ===
Parâmetros Treináveis : 14,086,144
Memória Estimada Modelo: 53.73 MB
Tempo de Inferência   : 0.1326 s

=== Exp 3: Aumento de Dimensão (256 emb / 8 heads) ===
Parâmetros Treináveis : 27,375,616
Memória Estimada Modelo: 104.43 MB
Tempo de Inferência   : 0.0993 s



In [8]:
import sys
from pathlib import Path
import torch

# Garantir a inclusão do diretório do código no caminho de busca
sys.path.append(str(Path.cwd().parent / "Codigo"))

# Importação de todos os módulos desenvolvidos no projeto
from bpe import BPETokenizer
from dataset import create_dataloader_v1
from embeddings import GPTInputEmbeddings
from attention import MultiHeadAttention
from model import GPTModel
from generation import generate_text_advanced, generate_text_simple

print("================================================================")
print("   TESTE DE INTEGRAÇÃO GLOBAL DA ARQUITETURA GPT (PIPELINE)    ")
print("================================================================\n")

# 1. TESTE DO TOKENIZER (BPE)
print("[1/6] Testando BPETokenizer...")
tokenizer = BPETokenizer(model_name="gpt2")
texto_exemplo = "Hello, GPT architecture integration test!"
token_ids = tokenizer.encode(texto_exemplo)
texto_recuperado = tokenizer.decode(token_ids)

print(f"   -> Texto de Entrada : '{texto_exemplo}'")
print(f"   -> IDs Codificados  : {token_ids}")
print(f"   -> Texto Decodificado: '{texto_recuperado}'")
assert texto_exemplo == texto_recuperado, "Erro no BPE Tokenizer!"
print("   [OK] Tokenizer validado.\n")


# 2. TESTE DO DATASET & DATALOADER
print("[2/6] Testando Dataset e DataLoader...")
texto_corpo = "The GPT model processes text autorregressively token by token using multi-head causal attention mechanism."
dataloader = create_dataloader_v1(
    txt=texto_corpo,
    batch_size=2,
    max_length=8,
    stride=4,
    shuffle=False
)
inputs_batch, targets_batch = next(iter(dataloader))
print(f"   -> Lote de Entrada (X) shape : {list(inputs_batch.shape)}")
print(f"   -> Lote de Alvos (Y) shape   : {list(targets_batch.shape)}")
assert inputs_batch.shape == (2, 8), "Erro no DataLoader!"
print("   [OK] Dataset e DataLoader validados.\n")


# 3. TESTE DOS EMBEDDINGS COMBINADOS
print("[3/6] Testando GPTInputEmbeddings...")
emb_layer = GPTInputEmbeddings(
    vocab_size=tokenizer.vocab_size,
    output_dim=64,
    max_length=256
)
embeddings_out = emb_layer(inputs_batch)
print(f"   -> Saída dos Embeddings shape: {list(embeddings_out.shape)} -> [batch, seq_len, emb_dim]")
assert embeddings_out.shape == (2, 8, 64), "Erro no módulo de Embeddings!"
print("   [OK] Embeddings combinados validados.\n")


# 4. TESTE DO MULTI-HEAD ATTENTION INDIVIDUAL
print("[4/6] Testando MultiHeadAttention isolado...")
mha = MultiHeadAttention(
    d_in=64,
    d_out=64,
    context_length=256,
    dropout=0.0,
    num_heads=4,
    qkv_bias=False
)
attn_out = mha(embeddings_out)
print(f"   -> Saída da Atenção shape    : {list(attn_out.shape)}")
assert attn_out.shape == embeddings_out.shape, "Erro de dimensão no MHA!"
print("   [OK] Multi-Head Attention isolado validado.\n")


# 5. TESTE DA ARQUITETURA COMPLETA (GPTModel)
print("[5/6] Testando GPTModel (Forward Pass Completo)...")
cfg = {
    "vocab_size": tokenizer.vocab_size,
    "context_length": 256,
    "emb_dim": 64,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.0,
    "qkv_bias": False
}
model = GPTModel(cfg)
model.eval()

with torch.no_grad():
    logits = model(inputs_batch)

print(f"   -> Entrada de Tokens shape   : {list(inputs_batch.shape)}")
print(f"   -> Saída de Logits shape     : {list(logits.shape)} -> [batch, seq_len, vocab_size]")
assert logits.shape == (2, 8, tokenizer.vocab_size), "Erro nos Logits do GPTModel!"
print("   [OK] Forward Pass do GPTModel validado.\n")


# 6. TESTE DA GERAÇÃO AUTORREGRESSIVA (FLUXO FINAL)
print("[6/6] Testando Geração Autorregressiva Texto -> Tokens -> Modelo -> Texto...")
prompt = "The Transformer model"
prompt_ids = torch.tensor([tokenizer.encode(prompt)])

out_ids_advanced = generate_text_advanced(
    model=model,
    idx=prompt_ids,
    max_new_tokens=6,
    context_size=cfg["context_length"],
    temperature=0.8,
    top_k=10
)

texto_gerado = tokenizer.decode(out_ids_advanced[0].tolist())

print(f"   -> Prompt Inicial            : '{prompt}'")
print(f"   -> Prompt Token IDs          : {prompt_ids.tolist()[0]}")
print(f"   -> IDs Finais com Geração   : {out_ids_advanced.tolist()[0]}")
print(f"   -> Texto Final Gerado        : '{texto_gerado}'")

assert out_ids_advanced.size(1) == prompt_ids.size(1) + 6, "Erro no tamanho do contexto gerado!"
print("\n================================================================")
print("   [SUCESSO INTEGRAL] TODOS OS COMPONENTES CONECTADOS E ATIVOS!   ")
print("================================================================")

   TESTE DE INTEGRAÇÃO GLOBAL DA ARQUITETURA GPT (PIPELINE)    

[1/6] Testando BPETokenizer...
   -> Texto de Entrada : 'Hello, GPT architecture integration test!'
   -> IDs Codificados  : [15496, 11, 402, 11571, 10959, 11812, 1332, 0]
   -> Texto Decodificado: 'Hello, GPT architecture integration test!'
   [OK] Tokenizer validado.

[2/6] Testando Dataset e DataLoader...
   -> Lote de Entrada (X) shape : [2, 8]
   -> Lote de Alvos (Y) shape   : [2, 8]
   [OK] Dataset e DataLoader validados.

[3/6] Testando GPTInputEmbeddings...
   -> Saída dos Embeddings shape: [2, 8, 64] -> [batch, seq_len, emb_dim]
   [OK] Embeddings combinados validados.

[4/6] Testando MultiHeadAttention isolado...
   -> Saída da Atenção shape    : [2, 8, 64]
   [OK] Multi-Head Attention isolado validado.

[5/6] Testando GPTModel (Forward Pass Completo)...
   -> Entrada de Tokens shape   : [2, 8]
   -> Saída de Logits shape     : [2, 8, 50257] -> [batch, seq_len, vocab_size]
   [OK] Forward Pass do GPTModel valida

In [9]:
import sys
from pathlib import Path
import time
import torch
import torch.nn.functional as F

# Garantir acesso aos módulos salvos na pasta 'Codigo'
sys.path.append(str(Path.cwd().parent / "Codigo"))

from model import GPTModel, TransformerBlock
from generation import generate_text_advanced, generate_text_simple

print("========================================================================")
print("  PIPELINE INTEGRADO: TRANSFORMER BLOCK -> GPT -> INFERÊNCIA AUTORREGRESSIVA")
print("========================================================================\n")

# ----------------------------------------------------------------------
# PASSO 1: CONFIGURAÇÃO DE PARÂMETROS
# ----------------------------------------------------------------------
cfg = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.0,
    "qkv_bias": False
}

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------------------
# PASSO 2: TESTE DO TRANSFORMER BLOCK
# ----------------------------------------------------------------------
print("[Passo 1] Testando a passagem individual pelo Transformer Block...")
block = TransformerBlock(cfg).to(device)
block.eval()

x_ficticio = torch.randn(2, 8, cfg["emb_dim"]).to(device)
with torch.no_grad():
    out_block = block(x_ficticio)

print(f"   -> Entrada do Bloco : {list(x_ficticio.shape)} [batch, seq_len, emb_dim]")
print(f"   -> Saída do Bloco   : {list(out_block.shape)} [batch, seq_len, emb_dim]")
assert out_block.shape == x_ficticio.shape, "Erro na simetria do Transformer Block!"
print("   [OK] Bloco Transformer validado sem perda dimensional.\n")

# ----------------------------------------------------------------------
# PASSO 3: CONSTRUÇÃO E FORWARD PASS DO MODELO GPT
# ----------------------------------------------------------------------
print("[Passo 2] Integrando os blocos no modelo GPT completo...")
model = GPTModel(cfg).to(device)
model.eval()

input_ids = torch.tensor([[101, 2054, 2003, 1037]], device=device) # Sequence len = 4
with torch.no_grad():
    logits = model(input_ids)

print(f"   -> Entradas de Token IDs : {list(input_ids.shape)}")
print(f"   -> Logits Gerados        : {list(logits.shape)} -> [batch, seq_len, vocab_size]")
assert logits.shape == (1, 4, cfg["vocab_size"]), "Erro na projeção de logits do modelo!"
print("   [OK] Mapeamento de Token IDs para Logits ativo.\n")

# ----------------------------------------------------------------------
# PASSO 4: ENTREGÁVEL 4 - CICLO DE INFERÊNCIA E GERAÇÃO AUTORREGRESSIVA
# ----------------------------------------------------------------------
print("[Passo 3] Executando o loop de Geração Autorregressiva...")
max_novos_tokens = 5
contexto_corrente = input_ids.clone()

print(f"   -> Contexto Inicial : {contexto_corrente.tolist()[0]}")

for passo in range(1, max_novos_tokens + 1):
    with torch.no_grad():
        # Restringe à janela máxima de contexto
        idx_cond = contexto_corrente[:, -cfg["context_length"]:]
        logits_step = model(idx_cond)
        
        # Foca estritamente no vetor da última posição
        logits_ultimo = logits_step[:, -1, :]
        
        # Ajuste de Temperatura e Top-K
        temp = 0.8
        top_k = 10
        top_logits, _ = torch.topk(logits_ultimo, top_k)
        logits_filtrados = torch.where(
            logits_ultimo < top_logits[:, -1:],
            torch.tensor(float('-inf'), device=device),
            logits_ultimo
        )
        
        probas = F.softmax(logits_filtrados / temp, dim=-1)
        prox_token = torch.multinomial(probas, num_samples=1)
        
        # O novo token alimenta a entrada da próxima iteração
        contexto_corrente = torch.cat((contexto_corrente, prox_token), dim=1)
        print(f"      Passo {passo}: Token {prox_token.item():<6} -> Sequência: {contexto_corrente.tolist()[0]}")

print("   [OK] Loop Autorregressivo concluído com sucesso.\n")

# ----------------------------------------------------------------------
# PASSO 5: ENTREGÁVEL 5 - EXPERIMENTAÇÃO E BENCHMARK
# ----------------------------------------------------------------------
print("[Passo 4] Executando o Benchmark dos Experimentos (Entregável 5)...")

def rodar_experimento_bench(nome_exp, cfg_exp, prompt_tensor):
    model_exp = GPTModel(cfg_exp).to(device)
    model_exp.eval()
    
    params = sum(p.numel() for p in model_exp.parameters() if p.requires_grad)
    mem_mb = (params * 4) / (1024 * 1024)
    
    inicio = time.time()
    with torch.no_grad():
        out = generate_text_advanced(
            model=model_exp,
            idx=prompt_tensor.to(device),
            max_new_tokens=10,
            context_size=cfg_exp["context_length"],
            temperature=0.8,
            top_k=10
        )
    tempo_exec = time.time() - inicio
    
    print(f"=== {nome_exp} ===")
    print(f"   Parâmetros Treináveis  : {params:,}")
    print(f"   Memória Estimada Modelo: {mem_mb:.2f} MB")
    print(f"   Tempo de Inferência    : {tempo_exec:.4f} s")

cfg_base = {"vocab_size": 50257, "context_length": 256, "emb_dim": 128, "n_heads": 4, "n_layers": 2, "drop_rate": 0.0, "qkv_bias": False}
cfg_deep = {"vocab_size": 50257, "context_length": 256, "emb_dim": 128, "n_heads": 4, "n_layers": 6, "drop_rate": 0.0, "qkv_bias": False}
cfg_wide = {"vocab_size": 50257, "context_length": 256, "emb_dim": 256, "n_heads": 8, "n_layers": 2, "drop_rate": 0.0, "qkv_bias": False}

rodar_experimento_bench("Exp 1: Base (Small)", cfg_base, input_ids)
rodar_experimento_bench("Exp 2: Profundo (6 Layers)", cfg_deep, input_ids)
rodar_experimento_bench("Exp 3: Largo (256 Emb / 8 Heads)", cfg_wide, input_ids)

print("\n========================================================================")
print("     [SUCESSO] PIPELINE COMPLETO INTEGRADO E EXECUTADO COM ÉXITO!       ")
print("========================================================================")

  PIPELINE INTEGRADO: TRANSFORMER BLOCK -> GPT -> INFERÊNCIA AUTORREGRESSIVA

[Passo 1] Testando a passagem individual pelo Transformer Block...
   -> Entrada do Bloco : [2, 8, 128] [batch, seq_len, emb_dim]
   -> Saída do Bloco   : [2, 8, 128] [batch, seq_len, emb_dim]
   [OK] Bloco Transformer validado sem perda dimensional.

[Passo 2] Integrando os blocos no modelo GPT completo...
   -> Entradas de Token IDs : [1, 4]
   -> Logits Gerados        : [1, 4, 50257] -> [batch, seq_len, vocab_size]
   [OK] Mapeamento de Token IDs para Logits ativo.

[Passo 3] Executando o loop de Geração Autorregressiva...
   -> Contexto Inicial : [101, 2054, 2003, 1037]
      Passo 1: Token 26268  -> Sequência: [101, 2054, 2003, 1037, 26268]
      Passo 2: Token 7855   -> Sequência: [101, 2054, 2003, 1037, 26268, 7855]
      Passo 3: Token 7331   -> Sequência: [101, 2054, 2003, 1037, 26268, 7855, 7331]
      Passo 4: Token 13971  -> Sequência: [101, 2054, 2003, 1037, 26268, 7855, 7331, 13971]
      Passo 